# Horse Racing RL — Phase 1 v9 Training

Incremental curriculum learning with the new drain-only stamina model.

**v9 changes over v8:**
- Drain-only stamina model (no recovery) with lateral movement costs
- `stamina_recovery` trait replaced by `drain_rate_mult` (0.7–1.3)
- Budget-aware reward function (penalizes burning faster than linear baseline)
- Progress-aware pacing: cruise bonus early, kick bonus late
- **Incremental stages**: each stage runs independently (~10-15 min), verify between stages
- Reduced timesteps: 2.8M baseline + 1.6M skill conditioning
- **Skill conditioning** (stages 9-12): 6 composable jockey skills as binary obs flags

**Workflow:** Run one stage cell at a time. Check the eval output before proceeding. If a stage fails its gate, re-run with more timesteps.

## 1. Setup

In [ ]:
import os

# Clone or pull latest
if os.path.exists("hr-simulation"):
    !cd hr-simulation && git pull
else:
    !git clone https://github.com/ue-too/hr-simulation.git

%cd hr-simulation

In [ ]:
# Remove old gym (Colab pre-installs it) and install dependencies
!pip uninstall -y gym 2>/dev/null
!pip install -q stable-baselines3 gymnasium pettingzoo numpy torch onnx onnxruntime tensorboard
# Install the project itself so scripts and imports work
!pip install -q -e .

In [ ]:
# Verify imports
from horse_racing.engine import HorseRacingEngine
from horse_racing.types import HorseAction, HORSE_COUNT, SPEED_DRAIN_RATE
from horse_racing.env import HorseRacingSingleEnv
from horse_racing.reward import compute_reward
print(f"Engine OK — {HORSE_COUNT} horses")
print(f"Stamina drain rate (distance tax): {SPEED_DRAIN_RATE}")

# Quick sanity check
engine = HorseRacingEngine("tracks/test_oval.json")
engine.step([HorseAction() for _ in range(HORSE_COUNT)])
obs = engine.get_observations()
arr = engine.obs_to_array(obs[0])
print(f"Observation vector: {arr.shape} — {arr.dtype}")
print(f"drain_rate_mult in obs: {obs[0]['drain_rate_mult']}")

## 2. Stamina Drain Sanity Check

Verify the drain constants produce sensible stamina curves before training. A cruising horse should finish a real track with some stamina remaining, while a max-effort horse should exhaust mid-race.

In [ ]:
!python scripts/check_stamina_drain.py --track tracks/tokyo.json

## 3. Training Utilities

In [ ]:
import os
import sys
import time
from collections import deque
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv

from horse_racing.env import HorseRacingSingleEnv

# ── Config ────────────────────────────────────────────────────────────
DEVICE = "cpu"  # MLP policies are faster on CPU
LOG_DIR = "logs/baseline"
VERSION = "v9"

cpu_count = os.cpu_count() or 2
N_ENVS = max(2, cpu_count)
BATCH_SIZE = max(256, N_ENVS * 32)

print(f"CPUs: {cpu_count} | Envs: {N_ENVS} | Batch: {BATCH_SIZE} | Device: {DEVICE}")

CHECKPOINT_DIR = Path("checkpoints/baseline")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = Path("models") / VERSION
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ── Stage definitions ────────────────────────────────────────────────
STAGES = [
    {"track": "tracks/curriculum_1_straight.json", "timesteps": 200_000, "max_steps": 1500, "gate": 0.90, "name": "Stage 1: Straight"},
    {"track": "tracks/curriculum_2_gentle_oval.json", "timesteps": 300_000, "max_steps": 7000, "gate": 0.80, "name": "Stage 2: Gentle oval"},
    {"track": "tracks/curriculum_3_tight_oval.json", "timesteps": 300_000, "max_steps": 4000, "gate": 0.70, "name": "Stage 3: Tight oval"},
    {"track": "tracks/tokyo.json", "timesteps": 400_000, "max_steps": 3500, "gate": 0.60, "name": "Stage 4: Tokyo"},
    {"track": "tracks/kokura.json", "timesteps": 400_000, "max_steps": 5500, "gate": 0.60, "name": "Stage 5: Kokura"},
    {"track": "tracks/tokyo_2600.json", "timesteps": 400_000, "max_steps": 6000, "gate": 0.50, "name": "Stage 6: Tokyo 2600"},
    {"track": "tracks/hanshin.json", "timesteps": 400_000, "max_steps": 4000, "gate": 0.50, "name": "Stage 7: Hanshin"},
    {"track": "tracks/kyoto.json", "timesteps": 400_000, "max_steps": 4000, "gate": 0.50, "name": "Stage 8: Kyoto"},
    # Skill conditioning stages
    {"track": "tracks/tokyo.json", "timesteps": 400_000, "max_steps": 3500, "gate": 0.60, "name": "Stage 9: Skills – Tokyo", "random_skills": True, "min_skills": 1, "max_skills": 2},
    {"track": "tracks/hanshin.json", "timesteps": 400_000, "max_steps": 4000, "gate": 0.50, "name": "Stage 10: Skills – Hanshin", "random_skills": True, "min_skills": 1, "max_skills": 3},
    {"track": "tracks/kokura.json", "timesteps": 400_000, "max_steps": 5500, "gate": 0.50, "name": "Stage 11: Skills – Kokura", "random_skills": True, "min_skills": 1, "max_skills": 3},
    {"track": "tracks/kyoto.json", "timesteps": 400_000, "max_steps": 4000, "gate": 0.50, "name": "Stage 12: Skills – Kyoto", "random_skills": True, "min_skills": 2, "max_skills": 3},
]

total_baseline = sum(s["timesteps"] for s in STAGES[:8])
total_skills = sum(s["timesteps"] for s in STAGES[8:])
print(f"Baseline: {total_baseline:,} timesteps (stages 1-8)")
print(f"Skills:   {total_skills:,} timesteps (stages 9-12)")
print(f"Total:    {total_baseline + total_skills:,} timesteps across {len(STAGES)} stages")


# ── ONNX export ──────────────────────────────────────────────────────
class PolicyNetwork(nn.Module):
    def __init__(self, sb3_policy):
        super().__init__()
        self.features_extractor = sb3_policy.features_extractor
        self.mlp_extractor = sb3_policy.mlp_extractor
        self.action_net = sb3_policy.action_net

    def forward(self, obs):
        features = self.features_extractor(obs)
        latent_pi, _ = self.mlp_extractor(features)
        return self.action_net(latent_pi)


def export_onnx(sb3_model, output_path):
    wrapper = PolicyNetwork(sb3_model.policy)
    wrapper.eval()
    obs_dim = sb3_model.observation_space.shape[0]
    dummy = torch.zeros(1, obs_dim, dtype=torch.float32)
    torch.onnx.export(
        wrapper, dummy, output_path,
        input_names=["obs"], output_names=["action"],
        dynamic_axes={"obs": {0: "batch"}, "action": {0: "batch"}},
        opset_version=17, dynamo=False,
    )


# ── Callbacks ─────────────────────────────────────────────────────────
class ProgressCallback(BaseCallback):
    def __init__(self, stage_name, total_timesteps, print_freq=4096):
        super().__init__()
        self.stage_name = stage_name
        self.total = total_timesteps
        self.print_freq = print_freq
        self.start_time = None

    def _on_training_start(self):
        self.start_time = time.time()
        self.step_offset = self.num_timesteps

    def _on_step(self) -> bool:
        if self.n_calls % self.print_freq == 0 and len(self.model.ep_info_buffer) > 0:
            mean_r = np.mean([ep["r"] for ep in self.model.ep_info_buffer])
            elapsed = time.time() - self.start_time
            stage_steps = self.num_timesteps - self.step_offset
            sps = stage_steps / elapsed if elapsed > 0 else 0
            pct = 100 * stage_steps / self.total
            eta = (self.total - stage_steps) / sps if sps > 0 else 0
            print(
                f"  [{self.stage_name}] {pct:5.1f}% | "
                f"steps: {stage_steps:>8,} | reward: {mean_r:8.2f} | "
                f"{sps:.0f} sps | ETA: {eta / 60:.1f}m"
            )
        return True


# ── Eval ──────────────────────────────────────────────────────────────
def run_eval(model, track_path, max_steps, episodes=3):
    completions, rewards, final_staminas, avg_speeds = [], [], [], []
    for ep in range(episodes):
        env = HorseRacingSingleEnv(track_path=track_path, max_steps=max_steps)
        obs, _ = env.reset()
        total_reward, speeds = 0.0, []
        for step in range(max_steps):
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            o = env.engine.get_observations()[0]
            speeds.append(o["tangential_vel"])
            if terminated or truncated:
                completions.append(o.get("finished", False))
                final_staminas.append(o["stamina_ratio"])
                break
        rewards.append(total_reward)
        avg_speeds.append(np.mean(speeds) if speeds else 0)
        env.close()
        sys.stdout.write("." if completions[-1] else "x")
        sys.stdout.flush()
    print()
    return {
        "completion_rate": np.mean(completions) if completions else 0,
        "mean_reward": np.mean(rewards),
        "mean_final_stamina": np.mean(final_staminas) if final_staminas else 0,
        "mean_avg_speed": np.mean(avg_speeds),
    }


# ── Train one stage ──────────────────────────────────────────────────
def make_env(track_path, max_steps, random_skills=False, min_skills=1, max_skills=3):
    def _init():
        return Monitor(HorseRacingSingleEnv(
            track_path=track_path, max_steps=max_steps,
            random_skills=random_skills, min_skills=min_skills, max_skills=max_skills,
        ))
    return _init


def train_stage(stage_num, timesteps_override=None):
    """Train a single stage. Returns (model, passed_gate)."""
    stage = STAGES[stage_num - 1]
    timesteps = timesteps_override or stage["timesteps"]

    print(f"{'=' * 60}")
    print(f"  {stage['name']} — {timesteps:,} timesteps")
    print(f"  track: {stage['track']} | gate: {stage['gate']:.0%}")
    if stage.get("random_skills"):
        print(f"  skills: random {stage['min_skills']}-{stage['max_skills']}")

    # Resume from previous stage if available
    resume_path = None
    if stage_num > 1:
        prev = CHECKPOINT_DIR / f"stage_{stage_num - 1}.zip"
        if prev.exists():
            resume_path = str(prev)
            print(f"  resume: {resume_path}")
    print(f"{'=' * 60}")

    env = SubprocVecEnv([
        make_env(
            stage["track"], stage["max_steps"],
            random_skills=stage.get("random_skills", False),
            min_skills=stage.get("min_skills", 1),
            max_skills=stage.get("max_skills", 3),
        ) for _ in range(N_ENVS)
    ])

    if resume_path:
        model = PPO.load(resume_path, env=env, device=DEVICE)
        model.tensorboard_log = LOG_DIR
    else:
        model = PPO(
            "MlpPolicy", env, verbose=0,
            n_steps=2048, batch_size=BATCH_SIZE, n_epochs=10,
            learning_rate=3e-4, gamma=0.99, gae_lambda=0.95,
            clip_range=0.2, vf_coef=0.5, ent_coef=0.01,
            device=DEVICE, tensorboard_log=LOG_DIR,
        )

    callback = ProgressCallback(stage["name"], timesteps)
    start = time.time()
    model.learn(
        total_timesteps=timesteps, callback=callback,
        reset_num_timesteps=(resume_path is None),
        tb_log_name=f"stage_{stage_num}",
    )
    elapsed = time.time() - start
    print(f"\n  Training done in {elapsed / 60:.1f} minutes")

    # Save checkpoint + ONNX
    save_path = CHECKPOINT_DIR / f"stage_{stage_num}"
    model.save(str(save_path))
    onnx_path = MODELS_DIR / f"baseline_stage{stage_num}.onnx"
    export_onnx(model, str(onnx_path))
    print(f"  Checkpoint → {save_path}.zip")
    print(f"  ONNX       → {onnx_path}")

    env.close()

    # Eval
    print(f"\n  Evaluating (3 episodes)...")
    sys.stdout.write("  ")
    stats = run_eval(model, stage["track"], stage["max_steps"], episodes=3)
    print(f"  Completion: {stats['completion_rate']:.0%}")
    print(f"  Mean reward: {stats['mean_reward']:.1f}")
    print(f"  Final stamina: {stats['mean_final_stamina']:.1%}")
    print(f"  Avg speed: {stats['mean_avg_speed']:.1f} m/s")

    passed = stats["completion_rate"] >= stage["gate"]
    if passed:
        print(f"\n  GATE PASSED ({stats['completion_rate']:.0%} >= {stage['gate']:.0%})")
    else:
        print(f"\n  GATE FAILED ({stats['completion_rate']:.0%} < {stage['gate']:.0%})")
        print(f"  Re-run: train_stage({stage_num}, timesteps_override={timesteps * 2})")

    return model, passed


print("Utilities ready")

## 4. Curriculum Stages

Run each stage one at a time. Check the eval output before proceeding to the next.
If a stage fails its gate, re-run with more timesteps: `train_stage(N, timesteps_override=600_000)`

In [ ]:
# Stage 1: Straight (200K steps, ~5 min)
model, passed = train_stage(1)

In [ ]:
# Stage 2: Gentle oval (300K steps, ~8 min)
model, passed = train_stage(2)

In [ ]:
# Stage 3: Tight oval (300K steps, ~8 min)
model, passed = train_stage(3)

In [ ]:
# Stage 4: Tokyo (400K steps, ~12 min)
model, passed = train_stage(4)

In [ ]:
# Stage 5: Kokura (400K steps, ~12 min)
model, passed = train_stage(5)

In [ ]:
# Stage 6: Tokyo 2600 (400K steps, ~12 min)
model, passed = train_stage(6)

In [ ]:
# Stage 7: Hanshin (400K steps, ~12 min)
model, passed = train_stage(7)

In [ ]:
# Stage 8: Kyoto (400K steps, ~12 min)
model, passed = train_stage(8)

## 5. Skill Conditioning Stages

Stages 9-12 train composable jockey skills on top of the baseline. Each episode randomly activates 1-3 skills from the 6 available:
- **stamina_management** — efficient stamina budgeting
- **sprint_timing** — late-race speed bursts
- **overtake** — closing and passing opponents
- **drafting_exploit** — exploit draft early, break away late
- **cornering_line** — inside lines on curves
- **pace_pressure** — sustained high speed

Skill flags are binary inputs in the observation vector (indices 102-107). The network learns conditioned behavior based on which skills are active.

In [ ]:
# Stage 9: Skills – Tokyo (400K steps, random 1-2 skills)
model, passed = train_stage(9)

In [ ]:
# Stage 10: Skills – Hanshin (400K steps, random 1-3 skills)
model, passed = train_stage(10)

In [ ]:
# Stage 11: Skills – Kokura (400K steps, random 1-3 skills)
model, passed = train_stage(11)

In [ ]:
# Stage 12: Skills – Kyoto (400K steps, random 2-3 skills)
model, passed = train_stage(12)

## 6. Verify ONNX Export

In [ ]:
import onnxruntime as ort
import glob

# Export final baseline model
final_onnx = str(MODELS_DIR / "baseline.onnx")
export_onnx(model, final_onnx)

# Verify
sess = ort.InferenceSession(final_onnx)
obs_dim = model.observation_space.shape[0]
test = np.zeros((1, obs_dim), dtype=np.float32)
result = sess.run(["action"], {"obs": test})
print(f"ONNX verified: {final_onnx}")
print(f"Input: (1, {obs_dim}) → Output: {result[0][0]}")

# List all exported stage models
stage_models = sorted(glob.glob(str(MODELS_DIR / "baseline_stage*.onnx")))
print(f"\nStage models ({len(stage_models)}):")
for p in stage_models:
    print(f"  {p}")

## 7. Quick Race Check

Run the trained model on Tokyo and watch stamina drain behavior.

In [ ]:
# Run a race with the trained model and check stamina behavior
engine = HorseRacingEngine("tracks/tokyo.json")
sess = ort.InferenceSession(final_onnx)

for tick in range(3000):
    obs = engine.get_observations()
    arr = engine.obs_to_array(obs[0]).reshape(1, -1)
    action = sess.run(["action"], {"obs": arr})[0][0]

    actions = [HorseAction(float(action[0]), float(action[1]))]
    for _ in range(1, HORSE_COUNT):
        actions.append(HorseAction())
    engine.step(actions)

    if tick % 500 == 499:
        o = obs[0]
        print(
            f"Tick {tick:4d} | "
            f"progress: {o['track_progress']:.3f} | "
            f"vel: {o['tangential_vel']:.1f} | "
            f"stamina: {o['stamina_ratio']:.3f} | "
            f"disp: {o['displacement']:+.2f} | "
            f"normal_vel: {o['normal_vel']:+.3f}"
        )

    if engine.horses[0].finished:
        o = obs[0]
        print(f"\nFinished at tick {tick}!")
        print(f"Final stamina: {engine.horses[0].runtime.current_stamina:.1f} / {engine.horses[0].base_attrs.stamina:.1f}")
        break

In [ ]:
# Compare behavior with different skill combos
from horse_racing.skills import ARCHETYPE_SKILL_PRESETS
from horse_racing.types import SKILL_IDS

final_onnx = str(MODELS_DIR / "baseline.onnx")
sess = ort.InferenceSession(final_onnx)

skill_combos = {
    "no_skills": set(),
    "closer": set(ARCHETYPE_SKILL_PRESETS["closer"]),
    "front_runner": set(ARCHETYPE_SKILL_PRESETS["front_runner"]),
    "all_skills": set(SKILL_IDS),
}

for combo_name, skills in skill_combos.items():
    env = HorseRacingSingleEnv(
        track_path="tracks/tokyo.json", max_steps=3500,
        active_skills=skills,
    )
    obs, _ = env.reset()
    speeds = []
    for step in range(3500):
        action = sess.run(["action"], {"obs": obs.reshape(1, -1)})[0][0]
        obs, reward, terminated, truncated, _ = env.step(action)
        speeds.append(env.engine.get_observations()[0]["tangential_vel"])
        if terminated or truncated:
            break
    o = env.engine.get_observations()[0]
    finished = "finished" if env.engine.horses[0].finished else "DNF"
    skills_str = str(sorted(skills)) if skills else "none"
    print(
        f"{combo_name:15s} | skills: {skills_str:40s} | "
        f"avg_vel: {np.mean(speeds):.1f} | stamina: {o['stamina_ratio']:.3f} | {finished}"
    )
    env.close()

## 8. Download Models

In [ ]:
import shutil

try:
    from google.colab import files
    # Download final model
    files.download(final_onnx)
    print(f"Downloading {final_onnx}")
    # Zip all stage models for bulk download
    shutil.make_archive("v9_stage_models", "zip", ".", str(MODELS_DIR))
    files.download("v9_stage_models.zip")
    print("Downloading v9_stage_models.zip")
except ImportError:
    print(f"Not in Colab. Models saved at: {MODELS_DIR}/")

In [ ]:
# Download TensorBoard logs for local viewing
# Run locally: tensorboard --logdir logs/baseline
shutil.make_archive("tb_logs_v9", "zip", ".", LOG_DIR)
try:
    from google.colab import files
    files.download("tb_logs_v9.zip")
except ImportError:
    print(f"Not in Colab. Logs at: {LOG_DIR}/")